# WattWise — Heuristic (Genetic Algorithm) for Battery Dispatch

Minimize the daily electricity bill by scheduling battery charging and discharging
in each 30-minute slot under CEB's Time-of-Use tariff.

**Method:** Genetic Algorithm (heuristic) — compared with the DP (exact method).

---

## Stage 1 — Data Loading

Load the cleaned 30-minute profiles into separate per-day arrays for solar, demand, and price.

### 1.1 Imports and data-loading functions

In [11]:
# import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [12]:
# Constants used throughout the notebook
DATA_PATH = Path("cleaning/profiles_30min.csv")
SLOTS_PER_DAY = 48 # 30-minute slots
SLOT_HOURS = 0.5 # each slot is half an hour


def load_profiles(path=DATA_PATH):
    """Load the cleaned 30-minute profile data."""
    df = pd.read_csv(path)
    df["production_kWh"] = df["production_kWh"].astype(float)
    df["consumption_kWh"] = df["consumption_kWh"].astype(float)
    df["price_LKR_per_kWh"] = df["price_LKR_per_kWh"].astype(float)
    return df


def list_days(df):
    """Return all available dates in the dataset."""
    return sorted(df["date"].unique())


def get_day(df, date):
    """Return one day's data as clean 48-element NumPy arrays."""
    d = (
        df[df["date"] == date]
        .sort_values("slot_start")
        .reset_index(drop=True)
    )

    if len(d) != SLOTS_PER_DAY:
        raise ValueError(
            f"{date}: expected {SLOTS_PER_DAY} slots, got {len(d)}"
        )

    return {
        "date": date,
        "solar": d["production_kWh"].to_numpy(), # kWh generated per slot
        "demand": d["consumption_kWh"].to_numpy(), # kWh consumed per slot
        "price": d["price_LKR_per_kWh"].to_numpy(), # Rs/kWh for that slot
        "band": d["band"].to_numpy(),
        "slot": d["slot_start"].to_numpy(),
    }

### 1.2 Sanity check — load one day and inspect it

In [13]:
# sanity check
df = load_profiles()
print("Days available:", list_days(df))

day = get_day(df, "2026-09-11")
print(f"\nLoaded {day['date']}")
print(f"  slots       : {len(day['solar'])}")
print(f"  total solar : {day['solar'].sum():.2f} kWh")
print(f"  total demand: {day['demand'].sum():.2f} kWh")
print(f"  price levels: {sorted(set(day['price']))} Rs/kWh")

Days available: ['2026-09-10', '2026-09-11', '2026-09-13', '2026-09-16', '2026-09-17']

Loaded 2026-09-11
  slots       : 48
  total solar : 14.07 kWh
  total demand: 9.58 kWh
  price levels: [np.float64(33.0), np.float64(47.0), np.float64(106.0)] Rs/kWh


### 1.3 Inspect slots, bands and prices
See how the 48 slots map to tariff bands and prices, and where the bands switch.

In [14]:
day = get_day(df, "2026-09-11")

# full 48-slot table: index, time, band, price
print("idx  time    band       price")
print("-" * 34)
for i in range(SLOTS_PER_DAY):
    print(f"{i:>3}  {day['slot'][i]:>5}   {day['band'][i]:<9}  {day['price'][i]:>5.0f}")

# where do the bands change? (should land on 05:30, 18:30, 22:30)
print("\nBand transitions:")
prev = None
for i in range(SLOTS_PER_DAY):
    if day["band"][i] != prev:
        print(f"  slot {i:>2} ({day['slot'][i]}) -> {day['band'][i]} (Rs {day['price'][i]:.0f})")
        prev = day["band"][i]

# how many slots in each band
print("\nSlots per band:")
bands, counts = np.unique(day["band"], return_counts=True)
for b, n in zip(bands, counts):
    print(f"  {b:<9}: {n} slots")

idx  time    band       price
----------------------------------
  0  00:00   Off-Peak      33
  1  00:30   Off-Peak      33
  2  01:00   Off-Peak      33
  3  01:30   Off-Peak      33
  4  02:00   Off-Peak      33
  5  02:30   Off-Peak      33
  6  03:00   Off-Peak      33
  7  03:30   Off-Peak      33
  8  04:00   Off-Peak      33
  9  04:30   Off-Peak      33
 10  05:00   Off-Peak      33
 11  05:30   Day           47
 12  06:00   Day           47
 13  06:30   Day           47
 14  07:00   Day           47
 15  07:30   Day           47
 16  08:00   Day           47
 17  08:30   Day           47
 18  09:00   Day           47
 19  09:30   Day           47
 20  10:00   Day           47
 21  10:30   Day           47
 22  11:00   Day           47
 23  11:30   Day           47
 24  12:00   Day           47
 25  12:30   Day           47
 26  13:00   Day           47
 27  13:30   Day           47
 28  14:00   Day           47
 29  14:30   Day           47
 30  15:00   Day           47
 31  

---

## Stage 2 — Battery Simulation & Bill Calculation

This is the core of the project: a function that takes a battery schedule (48 actions) and a battery size, simulates the battery throughout the day, and returns the electricity bill. This bill **is the objective function** that the GA will minimize.

**Sign convention:** positive action = discharge, negative = charge.


### 2.1 Battery constants (shared with the DP — keep identical)

In [15]:
# Battery model constants (MUST match Kasun's DP)
EFFICIENCY = 0.90 # round-trip loss, applied on charging only
MAX_RATE_KWH = 1.25 # max kWh moved per 30-min slot (~2.5 kW x 0.5 h)
START_SOC = 0.0 # battery starts empty each day
CAPACITIES = [0, 2, 4, 7] # capacity sweep (kWh)

# Export: surplus solar is NOT paid for (wasted) -> cost floored at 0

### 2.2 The simulation + bill function (the objective)

Converts a battery **schedule** (48 actions) into an **electricity bill** — this is the
objective function the GA minimizes. Walking slot by slot, it tracks the battery level
(SoC), checks the rules (rate, capacity), and sums the grid cost each slot.

**Returns:** `bill` (Rs), `violation` (0 if legal), `feasible` (bool), `soc` (battery level per slot).

In [16]:
def simulate(schedule, day, capacity,
             efficiency=EFFICIENCY, start_soc=START_SOC, max_rate=MAX_RATE_KWH):
    """
    Simulate one day's battery dispatch and return the bill and feasibility info.

    schedule : array of 48 actions (+ = discharge, - = charge), in kWh
    day      : dict with 'solar', 'demand', 'price' arrays (from get_day)
    capacity : battery size in kWh

    Returns dict: bill (Rs), violation (how far infeasible), feasible (bool),
                  soc (48-length state-of-charge trajectory)
    """
    schedule = np.asarray(schedule, dtype=float)
    solar, demand, price = day["solar"], day["demand"], day["price"]

    soc = start_soc
    soc_traj = np.zeros(SLOTS_PER_DAY)
    bill = 0.0
    violation = 0.0

    for t in range(SLOTS_PER_DAY):
        a = schedule[t]

        # Constraint 2: rate limit
        if abs(a) > max_rate:
            violation += abs(a) - max_rate

        # Constraint 3: SoC transition (loss on charge only)
        if a >= 0:  # discharge
            soc = soc - a
        else:       # charge
            soc = soc + efficiency * (-a)

        # Constraint 1: capacity limits
        if soc < 0:
            violation += -soc
        if soc > capacity:
            violation += soc - capacity
        soc_traj[t] = soc

        # Energy balance + cost (no export credit -> surplus wasted)
        grid = demand[t] - solar[t] - a
        bill += max(grid, 0.0) * price[t]

    return {
        "bill": bill,
        "violation": violation,
        "feasible": violation < 1e-9,
        "soc": soc_traj,
    }

In [17]:
day = get_day(df, "2026-09-11")

# run the simulation on a do-nothing schedule (should match the no-battery baseline)
result = simulate(np.zeros(SLOTS_PER_DAY), day, capacity=4)

# inspect what the function returns
print("Keys returned:", list(result.keys()))
print(f"  bill      : Rs {result['bill']:.2f}")
print(f"  violation : {result['violation']:.3f}")
print(f"  feasible  : {result['feasible']}")
print(f"  soc       : array of {len(result['soc'])} values (battery level each slot)")
print(f"  soc[:5]   : {result['soc'][:5]}")

Keys returned: ['bill', 'violation', 'feasible', 'soc']
  bill      : Rs 160.91
  violation : 0.000
  feasible  : True
  soc       : array of 48 values (battery level each slot)
  soc[:5]   : [0. 0. 0. 0. 0.]
